In [2]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

df = pd.read_csv('tokyo_weather.csv')

df_train = df.iloc[:-30].copy()
df_test = df.iloc[-30:].copy()




In [4]:
import numpy as np
import pandas as pd

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ---------------------------------------------------------
# Prepare data
# ---------------------------------------------------------

train_data = df_train[[
    "Highest Temperature (°C)",
    "Total Precipitation (mm)"
]].copy()

train_data["Highest Temperature (°C)"] = pd.to_numeric(
    train_data["Highest Temperature (°C)"],
    errors="coerce"
)

train_data["Total Precipitation (mm)"] = pd.to_numeric(
    train_data["Total Precipitation (mm)"],
    errors="coerce"
)

train_data = train_data.dropna().reset_index(drop=True)

# ---------------------------------------------------------
# Difference target variable
# ---------------------------------------------------------

y = train_data["Highest Temperature (°C)"].diff().dropna().reset_index(drop=True)

# ---------------------------------------------------------
# Create exogenous variables
# ---------------------------------------------------------

t = np.arange(len(train_data))

X = pd.DataFrame({
    "Total Precipitation (mm)": train_data["Total Precipitation (mm)"],
    "sin_year": np.sin(2 * np.pi * t / 365),
    "cos_year": np.cos(2 * np.pi * t / 365)
})

# IMPORTANT: remove first row so X matches differenced y
X = X.iloc[1:].reset_index(drop=True)

# ---------------------------------------------------------
# Time series cross-validation
# ---------------------------------------------------------

n_splits = 10
tscv = TimeSeriesSplit(n_splits=n_splits, test_size=1)

predictions = []
actuals = []
fold_numbers = []

for fold, (train_index, test_index) in enumerate(tscv.split(y), start=1):

    y_train_fold = y.iloc[train_index]
    y_test_fold = y.iloc[test_index]

    X_train_fold = X.iloc[train_index]
    X_test_fold = X.iloc[test_index]

    sarimax_model = SARIMAX(
        y_train_fold,
        order=(1, 1, 1),
        seasonal_order=(0, 0, 0, 0),
        exog=X_train_fold,
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    model_fit = sarimax_model.fit(disp=False)

    forecast_result = model_fit.get_forecast(
        steps=1,
        exog=X_test_fold
    )

    forecast_value = forecast_result.predicted_mean.iloc[0]

    predictions.append(forecast_value)
    actuals.append(y_test_fold.iloc[0])
    fold_numbers.append(fold)

# ---------------------------------------------------------
# Cross-validation results
# ---------------------------------------------------------

cv_results = pd.DataFrame({
    "Fold": fold_numbers,
    "Actual Difference": actuals,
    "Predicted Difference": predictions
})

cv_results["Error"] = cv_results["Actual Difference"] - cv_results["Predicted Difference"]
cv_results["Absolute Error"] = cv_results["Error"].abs()

display(cv_results)

mae_cv = mean_absolute_error(
    cv_results["Actual Difference"],
    cv_results["Predicted Difference"]
)

rmse_cv = np.sqrt(mean_squared_error(
    cv_results["Actual Difference"],
    cv_results["Predicted Difference"]
))

print("Cross-validation MAE on differenced temperature:", mae_cv)
print("Cross-validation RMSE on differenced temperature:", rmse_cv)

# ---------------------------------------------------------
# Fit final SARIMAX model
# ---------------------------------------------------------

final_model = SARIMAX(
    y,
    order=(1, 1, 1),
    seasonal_order=(0, 0, 0, 0),
    exog=X,
    enforce_stationarity=False,
    enforce_invertibility=False
)

final_fit = final_model.fit(disp=False)

print(final_fit.summary())

# ---------------------------------------------------------
# Last 15 in-sample predictions
# ---------------------------------------------------------

n_predictions = 15

prediction_result = final_fit.get_prediction(
    start=len(y) - n_predictions,
    end=len(y) - 1,
    exog=X.iloc[-n_predictions:]
)

predicted_diff = prediction_result.predicted_mean.reset_index(drop=True)
actual_diff = y.iloc[-n_predictions:].reset_index(drop=True)

# ---------------------------------------------------------
# Convert differenced predictions back to temperature levels
# ---------------------------------------------------------

start_temp = train_data["Highest Temperature (°C)"].iloc[-n_predictions - 1]

predicted_temp = start_temp + predicted_diff.cumsum()

actual_temp = train_data["Highest Temperature (°C)"].iloc[-n_predictions:].reset_index(drop=True)

# ---------------------------------------------------------
# Comparison table
# ---------------------------------------------------------

comparison_table = pd.DataFrame({
    "Actual Highest Temp": actual_temp,
    "Predicted Highest Temp": predicted_temp
})

comparison_table["Error"] = (
    comparison_table["Actual Highest Temp"] -
    comparison_table["Predicted Highest Temp"]
)

comparison_table["Absolute Error"] = comparison_table["Error"].abs()

comparison_table["Absolute Percentage Error"] = (
    comparison_table["Absolute Error"] /
    comparison_table["Actual Highest Temp"].abs()
) * 100

display(comparison_table)

# ---------------------------------------------------------
# Final accuracy
# ---------------------------------------------------------

mae_last_15 = comparison_table["Absolute Error"].mean()
mape_last_15 = comparison_table["Absolute Percentage Error"].mean()

print("MAE for last 15 highest temperature predictions:", mae_last_15)
print("MAPE for last 15 highest temperature predictions:", mape_last_15, "%")

C:\Users\GGPC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,Fold,Actual Difference,Predicted Difference,Error,Absolute Error
0,1,1.6,-0.105104,1.705104,1.705104
1,2,-4.7,0.033928,-4.733928,4.733928
2,3,-2.7,0.549995,-3.249995,3.249995
3,4,7.5,0.866053,6.633947,6.633947
4,5,-3.6,-1.566324,-2.033676,2.033676
5,6,-0.1,1.441456,-1.541456,1.541456
6,7,3.8,0.487883,3.312117,3.312117
7,8,-3.2,-0.569219,-2.630781,2.630781
8,9,0.8,1.332077,-0.532077,0.532077
9,10,-1.5,0.240678,-1.740678,1.740678


Cross-validation MAE on differenced temperature: 2.8113758477196122
Cross-validation RMSE on differenced temperature: 3.2803244545588326
                                  SARIMAX Results                                   
Dep. Variable:     Highest Temperature (°C)   No. Observations:                 2162
Model:                     SARIMAX(1, 1, 1)   Log Likelihood               -5605.213
Date:                      Fri, 01 May 2026   AIC                          11222.426
Time:                              17:12:45   BIC                          11256.490
Sample:                                   0   HQIC                         11234.885
                                     - 2162                                         
Covariance Type:                        opg                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------

,Actual Highest Temp,Predicted Highest Temp,Error,Absolute Error,Absolute Percentage Error
0,22.0,23.508461,-1.508461,1.508461,6.856639
1,21.6,24.171467,-2.571467,2.571467,11.904940
2,25.7,24.686535,1.013465,1.013465,3.943445
3,25.1,23.666054,1.433946,1.433946,5.712933
4,27.2,24.195153,3.004847,3.004847,11.047232
5,28.8,24.085370,4.714630,4.714630,16.370241
6,24.1,24.112625,-0.012625,0.012625,0.052386
7,21.4,24.661667,-3.261667,3.261667,15.241433
8,28.9,25.531610,3.368390,3.368390,11.655327
9,25.3,23.951494,1.348506,1.348506,5.330062


MAE for last 15 highest temperature predictions: 1.8573152566527642
MAPE for last 15 highest temperature predictions: 7.2517002515177635 %
